# Laptop Price Prediction



## 1. Importing Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np
import re

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ignore warning messages
import warnings
warnings.filterwarnings("ignore")

# Display plots inside notebook
%matplotlib inline

## 2. Loading the Dataset

In [ ]:
# Load the laptop dataset
df = pd.read_csv("laptops_uncleaned.csv")

## 3. Exploring the Dataset

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print("Shape:", df.shape)

In [ ]:
df.columns

In [ ]:
df.info()

## 4. Checking for Missing Values

In [ ]:
df.isnull().sum()

## 5. Statistical Summary

In [ ]:
df.describe()

## 6. Data Cleaning

In [ ]:
# Remove the unnecessary index column
df.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
# Remove the Indian Rupee symbol
df['price'] = df['price'].str.replace('₹', '', regex=False)

# Remove commas
df['price'] = df['price'].str.replace(',', '', regex=False)

# Convert the price column to a numeric data type
df['price'] = pd.to_numeric(df['price'])

## 7. Checking the Cleaned Price

In [ ]:
# Display the first 10 prices
df['price'].head(10)

## 8. Laptop Prices

In [ ]:
# Plot the distribution of laptop prices
plt.figure(figsize=(8, 5))

sns.histplot(df['price'], bins=30, kde=True)

plt.title('Laptop Price Distribution')
plt.xlabel('Price ')
plt.ylabel('Number of Laptops')

plt.show()

## 9. Rating vs Price

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(x='rating', y='price', data=df)

plt.title('Laptop Rating vs Price')
plt.xlabel('Rating')
plt.ylabel('Price')

plt.show()

## 10. Correlation Analysis

## 11. Exploring Laptop Features

In [ ]:
df['ram'].value_counts().head(10)

## 12. Cleaning RAM

In [ ]:
# Extract the RAM number from the beginning of the value
df['ram'] = df['ram'].str.extract(r'(\d+)')

# Convert RAM to a numeric value
df['ram'] = pd.to_numeric(df['ram'])

In [ ]:
df['ram'].head()

## 13. RAM vs Price

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(x='ram', y='price', data=df)

plt.title('RAM vs Laptop Price')
plt.xlabel('RAM (GB)')
plt.ylabel('Price ')

plt.show()

## 14. Memory

In [ ]:
df['memory'].value_counts().head(15)

## 15. Laptop Brand Analysis

In [ ]:
# Extract the first word from the laptop model as the brand
df['brand'] = df['model'].str.split().str[0]

# Display the most common brands
df['brand'].value_counts().head(15)

In [ ]:
# Calculate the average price for each brand
brand_price = df.groupby('brand')['price'].mean().sort_values(ascending=False)

brand_price.head(10)

## 16. Average Laptop Price by Brand

In [ ]:
plt.figure(figsize=(10, 6))

brand_price.head(10).plot(kind='bar')

plt.title('Average Laptop Price by Brand')
plt.xlabel('Brand')
plt.ylabel('Average Price ')

plt.xticks(rotation=45)
plt.show()

## 17. Feature Engineering 


In [ ]:
# storage
def parse_storage(value):
    if not isinstance(value, str):
        return np.nan
    match = re.search(r'(\d+)\s*(TB|GB)', value, re.IGNORECASE)
    if not match:
        return np.nan
    amount, unit = float(match.group(1)), match.group(2).upper()
    return amount * 1024 if unit == 'TB' else amount  # normalize everything to GB

df['storage_gb'] = df['memory'].apply(parse_storage)

In [ ]:
def storage_type(value):
    if not isinstance(value, str):
        return 'Other'
    value = value.upper()
    if 'SSD' in value:
        return 'SSD'
    if 'EMMC' in value:
        return 'eMMC'
    if 'HARD DISK' in value or 'HDD' in value:
        return 'HDD'
    return 'Other'

df['storage_type'] = df['memory'].apply(storage_type)
df['storage_type'].value_counts()

In [ ]:
def core_count(value):
    if not isinstance(value, str):
        return np.nan
    match = re.match(r'(\d+)', value.strip())
    if match:
        return int(match.group(1))
    word_map = {'dual': 2, 'quad': 4, 'hexa': 6, 'octa': 8}
    for word, count in word_map.items():
        if word in value.lower():
            return count
    return np.nan

df['core_count'] = df['core'].apply(core_count)

In [ ]:
def processor_brand(value):
    if not isinstance(value, str):
        return 'Other'
    value = value.lower()
    if 'intel' in value:
        return 'Intel'
    if 'amd' in value or 'ryzen' in value:
        return 'AMD'
    if 'apple' in value or 'm1' in value or 'm2' in value or 'm3' in value:
        return 'Apple'
    if 'qualcomm' in value or 'snapdragon' in value:
        return 'Qualcomm'
    return 'Other'

df['processor_brand'] = df['processor'].apply(processor_brand)
df['processor_brand'].value_counts()

In [ ]:
def has_dedicated_gpu(value):
    if not isinstance(value, str):
        return 0
    value = value.upper()
    return int(any(keyword in value for keyword in ['RTX', 'GTX', 'RADEON RX', 'MX']))

df['has_dedicated_gpu'] = df['graphic_card'].apply(has_dedicated_gpu)
df['has_dedicated_gpu'].value_counts()

## 17b. Additional Feature Engineering (Screen, GPU Memory, OS, Warranty)

The raw dataset also has `display`, `os`, and `warrenty` columns that weren't used yet, plus `graphic_card` still had unused detail (its VRAM size). These plausibly affect price too, so we extract them.

In [ ]:
# Screen size (inches) and resolution (total pixels) from 'display'
def parse_screen_size(value):
    if not isinstance(value, str):
        return np.nan
    match = re.search(r'([\d.]+)\s*inch', value, re.IGNORECASE)
    return float(match.group(1)) if match else np.nan

def parse_resolution(value):
    if not isinstance(value, str):
        return np.nan
    match = re.search(r'(\d+)\s*x\s*(\d+)', value, re.IGNORECASE)
    if not match:
        return np.nan
    width, height = int(match.group(1)), int(match.group(2))
    return width * height

df['screen_size'] = df['display'].apply(parse_screen_size)
df['screen_resolution'] = df['display'].apply(parse_resolution)

# GPU VRAM in GB from 'graphic_card' (0 for integrated/no dedicated GPU)
def parse_gpu_vram(value):
    if not isinstance(value, str):
        return 0
    match = re.match(r'(\d+)\s*GB', value.strip(), re.IGNORECASE)
    return int(match.group(1)) if match else 0

df['gpu_vram_gb'] = df['graphic_card'].apply(parse_gpu_vram)

# Operating system category from 'os'
def os_category(value):
    if not isinstance(value, str):
        return 'Other'
    value = value.lower()
    if 'windows' in value:
        return 'Windows'
    if 'mac' in value:
        return 'macOS'
    if 'dos' in value or 'no os' in value:
        return 'No OS'
    if 'linux' in value or 'chrome' in value:
        return 'Linux/Chrome'
    return 'Other'

df['os_category'] = df['os'].apply(os_category)

# Warranty length in years from 'warrenty'
def parse_warranty(value):
    if not isinstance(value, str):
        return np.nan
    match = re.search(r'(\d+)\s*Year', value, re.IGNORECASE)
    return int(match.group(1)) if match else np.nan

df['warranty_years'] = df['warrenty'].apply(parse_warranty)
# Only a handful of missing warranty values - fill with the most common warranty length instead of dropping rows
df['warranty_years'] = df['warranty_years'].fillna(df['warranty_years'].mode()[0])

df[['screen_size', 'screen_resolution', 'gpu_vram_gb', 'os_category', 'warranty_years']].describe(include='all')

In [ ]:
# Drop rows where key engineered features could not be parsed
df_model = df.dropna(subset=['rating', 'ram', 'storage_gb', 'core_count', 'screen_size', 'screen_resolution', 'price']).copy()
print(f"Rows kept: {len(df_model)} / {len(df)}")

## 17c. Removing Price Outliers

A handful of high-end gaming/workstation laptops sit far above the typical price range and pull the regression fit toward them, which is what was distorting the actual-vs-predicted plot. We drop rows outside the IQR (interquartile range) bound on `price`. This is done **before the train/test split**, since removing outliers is a data-cleaning decision, not something that should differ between train and test.

In [ ]:
# Remove price outliers using the IQR method
Q1 = df_model['price'].quantile(0.25)
Q3 = df_model['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

before = len(df_model)
df_model = df_model[(df_model['price'] >= lower_bound) & (df_model['price'] <= upper_bound)].copy()
after = len(df_model)

print(f"Price range kept: {lower_bound:,.0f} - {upper_bound:,.0f}")
print(f"Removed {before - after} outlier rows ({(before - after) / before:.1%} of data)")
print(f"Rows remaining: {after}")

In [ ]:
# Group rare brands into 'Other' 
top_brands = df_model['brand'].value_counts().head(10).index
df_model['brand_grouped'] = df_model['brand'].where(df_model['brand'].isin(top_brands), 'Other')
df_model['brand_grouped'].value_counts()

In [ ]:
# Correlation Heatmap for all numerical features

heatmap_cols = [
    'price',
    'rating',
    'ram',
    'storage_gb',
    'core_count',
    'has_dedicated_gpu',
    'gpu_vram_gb',
    'screen_size',
    'screen_resolution',
    'warranty_years'
]

correlation = df_model[heatmap_cols].corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)

plt.title('Correlation Heatmap - Price and Laptop Specifications')
plt.tight_layout()
plt.show()

## 18. Preparing the Data for ML

In [ ]:
# One-hot encode the categorical features and combine with the numeric ones
categorical_cols = ['brand_grouped', 'storage_type', 'processor_brand', 'os_category']
numeric_cols = ['rating', 'ram', 'storage_gb', 'core_count', 'has_dedicated_gpu', 'gpu_vram_gb', 'screen_size', 'screen_resolution', 'warranty_years']

X = pd.get_dummies(df_model[numeric_cols + categorical_cols], columns=categorical_cols, drop_first=True)

# Select the target we want to predict
y = df_model['price']

print("Number of features used:", X.shape[1])
X.head()

## 19. Splitting the Data

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## 19b. Normalizing Price and Rating (After the Split)

We fit the scalers **only on the training data** and reuse them to transform the test data. This avoids data leakage, since the model must never learn anything (even indirectly, like mean/std) from the test set.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Fit scalers on the TRAINING data only, then apply to both train and test
rating_scaler = StandardScaler()
price_scaler = StandardScaler()

# Normalize 'rating' feature
X_train['rating'] = rating_scaler.fit_transform(X_train[['rating']])
X_test['rating'] = rating_scaler.transform(X_test[['rating']])

# Normalize 'price' target
y_train_scaled = price_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test_scaled = price_scaler.transform(y_test.values.reshape(-1, 1)).flatten()

print("Rating (train) mean/std after scaling:", X_train['rating'].mean().round(3), X_train['rating'].std().round(3))
print("Price (train) mean/std after scaling:", y_train_scaled.mean().round(3), y_train_scaled.std().round(3))

## 20. Linear Regression

In [ ]:
# Create the Linear Regression model
lr_model = LinearRegression()

# Train the model using the scaled training target
lr_model.fit(X_train, y_train_scaled)

# Predict laptop prices (scaled) using the test data, then convert back to real prices
y_pred_lr_scaled = lr_model.predict(X_test)
y_pred_lr = price_scaler.inverse_transform(y_pred_lr_scaled.reshape(-1, 1)).flatten()

In [ ]:
# Evaluating Linear Regression
# Calculate the evaluation metrics for Linear Regression
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression")
print("MAE:", mae_lr)
print("RMSE:", rmse_lr)
print("R² Score:", r2_lr)

## 21. Random Forest Regression

In [ ]:
# Create the Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train the model on the scaled target
rf_model.fit(X_train, y_train_scaled)

# Predict laptop prices (scaled), then convert back to real prices
y_pred_rf_scaled = rf_model.predict(X_test)
y_pred_rf = price_scaler.inverse_transform(y_pred_rf_scaled.reshape(-1, 1)).flatten()

In [ ]:
# Evaluating the Random Forest Model
# Calculate the evaluation metrics for Random Forest
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R² Score:", r2_rf)

## 22. Gradient Boosting Regression

In [ ]:
# Create the Gradient Boosting model
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    random_state=42
)

# Train the model on the scaled target
gb_model.fit(X_train, y_train_scaled)

# Predict laptop prices (scaled), then convert back to real prices
y_pred_gb_scaled = gb_model.predict(X_test)
y_pred_gb = price_scaler.inverse_transform(y_pred_gb_scaled.reshape(-1, 1)).flatten()

In [ ]:
# Evaluating Gradient Boosting
# Calculate the evaluation metrics for Gradient Boosting
mae_gb = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print("Gradient Boosting")
print("MAE:", mae_gb)
print("RMSE:", rmse_gb)
print("R² Score:", r2_gb)

## 23. Models' Comparison

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [mae_lr, mae_rf, mae_gb],
    "RMSE": [rmse_lr, rmse_rf, rmse_gb],
    "R² Score": [r2_lr, r2_rf, r2_gb]
})

results

In [ ]:
# Plot the R² score for each model
plt.figure(figsize=(8, 5))

sns.barplot(x='Model', y='R² Score', data=results)

plt.title('Model Comparison - R² Score')
plt.xlabel('Model')
plt.ylabel('R² Score')

plt.xticks(rotation=15)
plt.show()

## 24. Actual vs Predicted Prices

In [ ]:
# Plot actual prices against predicted prices 
plt.figure(figsize=(8, 5))

plt.scatter(y_test, y_pred_gb)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect prediction')

plt.title('Actual vs Predicted Laptop Prices (Gradient Boosting)')
plt.xlabel('Actual Price ')
plt.ylabel('Predicted Price ')
plt.legend()

plt.show()
# momkem a delete el ouliers

## 25. Feature Importance 


In [ ]:
importances = pd.Series(gb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.head(10).plot(kind='barh')
plt.title('Top 10 Feature Importances - Gradient Boosting')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.show()

importances.head(10)